In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
# Preset values

years = [2025, 2026]
strt_month = 5

res_prop = "Residential"
res_sub = "SingleFamilyResidence"

crit_cols = ["ListPrice", "ListingKey", "ListingContractDate", "PurchaseContractDate",
             "CloseDate", "ClosePrice", "Latitude", "Longitude", "PostalCode",
             "PropertyType", "BedroomsTotal", "BathroomsTotalInteger", "LivingArea",
             "LotSizeSquareFeet", "DaysOnMarket", "UnparsedAddress"]                        # Define list of critical columns

sub_cols = ["ListPrice", "ListingKey", "ClosePrice", "Latitude", "Longitude",
            "PostalCode", "PropertyType", "CloseDate", "BedroomsTotal",
            "BathroomsTotalInteger", "LivingArea", "LotSizeSquareFeet", "DaysOnMarket"]     # Define subset columns

In [3]:
def load_data(yrs=years, strt=strt_month):
  """
  Takes an inputted list of years and integer-valued start month for the CRMLS files
  Returns merged DataFrame of read csvs
  """
  main_df = pd.DataFrame()                                              # Defines blank DataFrame
  months = [str(i) if i > 9 else "0"+str(i) for i in range (1,13)]      # Defines proper format and type of month values

  for a in range(strt-1, 12):           # Iterates through months, starting with inputted strt month
    try:
      df = pd.read_csv("CRMLSSold"+str(yrs[0])+months[a]+".csv", low_memory=False)    # Reads csv
      main_df = pd.concat([main_df, df], ignore_index=True)                           # Appends DataFrame of csv to "main_df"
    except:
      df = pd.read_csv("CRMLSSold"+str(yrs[0])+months[a]+"_filled.csv", low_memory=False)
      main_df = pd.concat([main_df, df], ignore_index=True)

  if len(yrs) > 2:                    # If "yrs" list is greater than two,
    for b in range(1,len(yrs)-1):       # For each year in "yrs", except the first and last
      for c in months:                    # Iterates through each month in "months" (whole year for yrs[b])
        try:
          df = pd.read_csv("CRMLSSold"+str(yrs[b])+c+".csv", low_memory=False)        # Reads csv
          main_df = pd.concat([main_df, df], ignore_index=True)                       # Appends DataFrame of csv to "main_df"
        except:
          df = pd.read_csv("CRMLSSold"+str(yrs[b])+c+"_filled.csv", low_memory=False)
          main_df = pd.concat([main_df, df], ignore_index=True)

  elif len(yrs) == 2:           # Elif,
    for d in range(0, strt):      # Iterates through months upto the strt for the last year in "yrs"
      try:
        df = pd.read_csv("CRMLSSold"+str(yrs[len(yrs)-1])+months[d]+".csv", low_memory=False)   # Reads csv
        main_df = pd.concat([main_df, df], ignore_index=True)                                   # Appends DataFrame of csv to "main_df"
      except:
        df = pd.read_csv("CRMLSSold"+str(yrs[len(yrs)-1])+months[d]+"_filled.csv", low_memory=False)
        main_df = pd.concat([main_df, df], ignore_index=True)

  return main_df

In [4]:
def clean_data(df, crit=crit_cols):
  """
  Takes a DataFrame and list of critical columns
  Returns a DataFrame with duplicate values dropped, as well as null and zeros based on subsets of columns
  """
  clean_df = df[(df["ClosePrice"]>0) & (df["LivingArea"]>0) & (df["BathroomsTotalInteger"]>0) & (df["DaysOnMarket"]>0)]
    # ListPrice, Latitude/Longitude, BedroomsTotal
  clean_df = clean_df.drop_duplicates()
  clean_df = clean_df.dropna(subset=crit, ignore_index=True)
  return clean_df

In [5]:
def encode_and_subset(df, prop=res_prop, sub_prop=res_sub, cols=sub_cols):
  """
  Takes the DataFrame, PropertyType value (str), PropertySubType value (str), and list of columns for subset DataFrame
  Returns subset of inputted DataFrame, based on inputted parameters
  """
  df = df[(df["PropertyType"] == prop)&(df["PropertySubType"] == sub_prop)]     # Takes subset of DataFrame based on property type and subtype
  sub_df = df[cols].reset_index(drop=True)                                      # Takes subset of columns for DataFrame

  sub_df["CloseDate"] = pd.to_datetime(sub_df["CloseDate"])     # Converts "CloseDate" values to datetime type
  zips = [int(str(a)[0:5]) for a in sub_df['PostalCode'].to_list()]    # Converts "PostalCode" values to int type
  sub_df['PostalCode'] = pd.DataFrame({"PostalCode": zips})        # Replaces "PostalCode" values with int type

  sub_df = pd.get_dummies(sub_df, columns=["PropertyType"], dtype=int)      # Encoded "PropertyType" column of the df
  enc_col = sub_df.columns[sub_df.shape[1]-1]                               # Defines the encoded column name
  sub_df.rename(columns={enc_col: enc_col.split("_")[1]}, inplace=True)     # Renames the encoded column

  return sub_df

In [6]:
def save_csv(df, filename):
  """
  Takes a DataFrame
  Saves the inputted DataFrame as a .csv file, given inputted name
  """
  df.to_csv(filename, index=False)

In [ ]:
## Normalization (MMS)

In [7]:
def test_train_split(df):
  """
  Takes a DataFrame
  Encodes "PropertyType" column
  Returns a defined training and test set for the DataFrame
  """
  yr_mo = []
  for i in df['CloseDate']:                                         # For each date in the "CloseDate" column
    yr, mo = i.year, i.month                                          # Define the year and month values of date i
    yr_mo.append([yr,mo])                                             # Append to "yr_mo" a list of date i's year and month
  te_set = [b for b in range(len(yr_mo)) if yr_mo[b] == [2026,5]]   # Define a list of row #s with date 05/2026
  te_rng = te_set[0::len(te_set)-1]                                 # Define a list of the first and last row in "te_set"
  tr, te = df[0:te_rng[0]], df[te_rng[0]:te_rng[1]]             # Define the training and test sets of the inputted df

  return tr, te

In [8]:
def main():
  main_df = load_data()
  clean = clean_data(main_df)
  sub = encode_and_subset(clean)
  save_csv(sub, "CRMLS_0525-0526_clean.csv")

  train, test = test_train_split(sub)

In [9]:
if __name__ == "__main__":
  main()